<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260408.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
%%bash

node -v
npm -v

v20.19.0
10.8.2


In [41]:
%%bash

npx create-vite@latest bootcamp --template react

└  Operation cancelled



In [42]:
%%bash

cd bootcamp

In [43]:
%%bash

cd /content/bootcamp
npm install


up to date, audited 158 packages in 986ms

38 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities


In [44]:
%%bash

nohup npm run dev -- --host 0.0.0.0 > vite.log 2>&1 &

In [45]:
%%bash

npm install express && npm install -g cloudflared


up to date, audited 66 packages in 873ms

22 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities

changed 1 package in 2s


In [46]:
%%bash

nohup cloudflared tunnel --url http://localhost:5173 > tunnel.log 2>&1 &

In [47]:
%%bash

tail -n 20 tunnel.log



---



In [48]:
%%bash

cd /content/bootcamp

cat << 'EOF' > vite.config.js
import { defineConfig } from 'vite'
import react from '@vitejs/plugin-react'

export default defineConfig({
  plugins: [react()],
  server: {
    host: true,
    allowedHosts: [
      '.trycloudflare.com'
    ]
  }
})
EOF

In [49]:
%%bash

# vite 프로세스 종료
kill $(lsof -t -i:5173) 2>/dev/null || true

cd /content/bootcamp
nohup npm run dev -- --host 0.0.0.0 > vite.log 2>&1 &
sleep 5
cat vite.log


> bootcamp@0.0.0 dev
> vite --host 0.0.0.0


  VITE v8.0.7  ready in 439 ms

  ➜  Local:   http://localhost:5173/
  ➜  Network: http://172.28.0.12:5173/


In [50]:
!nohup cloudflared tunnel --url http://localhost:5173 > tunnel.log 2>&1 &
!sleep 5
!grep -o 'https://.*\.trycloudflare\.com' tunnel.log

https://tropical-went-combined-tulsa.trycloudflare.com




---



In [51]:
%%writefile /content/bootcamp/src/Welcome.jsx

/**
 * 15.1 기본 구조 설명
 */

// Welcome이라는 이름의 함수형 컴포넌트를 정의합니다.
// React 컴포넌트 이름은 반드시 대문자로 시작해야 합니다.
function Welcome() {
  // return 키워드 뒤에 화면에 그려질 UI(JSX)를 작성합니다.
  return (
    // HTML처럼 보이지만 JavaScript의 확장 문법인 JSX입니다.
    <h1>안녕하세요</h1>
  );
}

// 이 파일을 다른 파일(예: App.js)에서 불러와서 사용할 수 있도록 내보냅니다.
export default Welcome;

Overwriting /content/bootcamp/src/Welcome.jsx


In [52]:
%%writefile /content/bootcamp/src/App.jsx

import { useState } from 'react';

function App() {
  // 검색어 state — input 입력값 실시간 저장
  const [keyword, setKeyword] = useState('');
  // 로딩 상태 state — true: 로딩 중 / false: 대기
  const [loading, setLoading] = useState(false);
  // 목록 state — 나중에 검색 결과 배열로 채울 공간
  const [list, setList] = useState([]);

  return (
    <div>
      {/* 키 입력마다 setKeyword 실행 -> keyword state 갱신 */}
      <input onChange={e => setKeyword(e.target.value)} placeholder="검색어" />

      {/* 클릭마다 loading을 반대값으로 토글 (!loading) */}
      <button onClick={() => setLoading(!loading)}>로딩 변경</button>

      {/* keyword state 그대로 출력 */}
      <p>검색어: {keyword}</p>

      {/* loading이 true면 '로딩 중', false면 '대기' 출력 */}
      <p>로딩 상태: {loading ? '로딩 중' : '대기'}</p>

      {/* list 배열의 길이 출력 — 현재는 빈 배열이라 0 */}
      <p>목록 길이: {list.length}</p>
    </div>
  );
}

export default App;

Overwriting /content/bootcamp/src/App.jsx


In [53]:
%%writefile /content/bootcamp/src/App.jsx

import { useReducer } from 'react';

// reducer: state와 action을 받아서 새로운 state를 반환하는 함수
// useState보다 복잡한 상태 로직을 한 곳에서 관리할 때 사용
function reducer(state, action) {
  switch (action.type) {
    case 'INCREASE':
      return { count: state.count + 1 }; // 현재값 +1 한 새 state 반환
    case 'DECREASE':
      return { count: state.count - 1 }; // 현재값 -1 한 새 state 반환
    case 'RESET':
      return { count: 0 };               // 0으로 초기화한 새 state 반환
    default:
      return state;                       // 알 수 없는 action이면 기존 state 그대로 반환
  }
}

function App() {
  // state: 현재 상태값 ({ count: 0 }으로 시작)
  // dispatch: action을 reducer로 보내는 함수 (setter 역할)
  // 두 번째 인자 { count: 0 } -> 초기값
  const [state, dispatch] = useReducer(reducer, { count: 0 });

  return (
    <div>
      {/* state.count -> reducer가 관리하는 현재 count값 출력 */}
      <h1>{state.count}</h1>

      {/* dispatch로 action 객체 전달 -> reducer의 switch문 실행 */}
      <button onClick={() => dispatch({ type: 'INCREASE' })}>증가</button>
      <button onClick={() => dispatch({ type: 'DECREASE' })}>감소</button>
      <button onClick={() => dispatch({ type: 'RESET' })}>초기화</button>
    </div>
  );
}

export default App;

Overwriting /content/bootcamp/src/App.jsx


In [54]:
%%writefile /content/bootcamp/src/App.jsx

import { useReducer, useState } from 'react';

// reducer: 리스트(배열) 상태 관리
// state = 현재 목록 배열, action.payload = 전달된 데이터
function reducer(state, action) {
  switch (action.type) {
    case 'ADD':
      // 기존 배열에 새 항목 추가한 새 배열 반환
      return [...state, action.payload];
    case 'REMOVE':
      // payload로 받은 인덱스(i)만 제외하고 나머지로 새 배열 반환
      return state.filter((_, i) => i !== action.payload);
    default:
      return state;
  }
}

function App() {
  // text: 입력창 값 관리 (useState — 단순 문자열이라 useState로 충분)
  const [text, setText] = useState('');
  // list: 목록 배열 관리 (useReducer — ADD/REMOVE 두 가지 동작이 있어서 useReducer 사용)
  const [list, dispatch] = useReducer(reducer, []); // 초기값 빈 배열

  return (
    <div>
      {/* 입력마다 text state 갱신 */}
      <input onChange={e => setText(e.target.value)} value={text} />

      <button
        onClick={() => {
          dispatch({ type: 'ADD', payload: text }); // text를 리스트에 추가
          setText('');                               // 추가 후 입력창 초기화
        }}
      >
        추가
      </button>

      {/* list 배열을 순회하며 항목 렌더링 */}
      {list.map((item, i) => (
        <div key={i}>
          <span>{item}</span>
          {/* 클릭한 항목의 인덱스(i)를 payload로 전달 -> reducer에서 해당 항목 제거 */}
          <button onClick={() => dispatch({ type: 'REMOVE', payload: i })}>
            삭제
          </button>
        </div>
      ))}
    </div>
  );
}

export default App;

Overwriting /content/bootcamp/src/App.jsx


In [55]:
%%writefile /content/bootcamp/src/App.jsx

import { createContext, useContext, useState } from 'react';

const ThemeContext = createContext();

function Header() {
  const theme = useContext(ThemeContext);

  return <h1>현재 테마: {theme}</h1>;
}

function App() {
  const [theme, setTheme] = useState('light');

  return (
    <ThemeContext.Provider value={theme}>
      <Header />
      <button onClick={() => setTheme('dark')}>테마 변경</button>
    </ThemeContext.Provider>
  );
}

export default App;

Overwriting /content/bootcamp/src/App.jsx


In [56]:
%%writefile /content/bootcamp/src/App.jsx

import { useState } from 'react';

function useToggle(initialValue = false) {
  const [value, setValue] = useState(initialValue);

  function toggle() {
    setValue(prev => !prev);
  }

  return [value, toggle];
}

function App() {
  const [isOpen, toggle] = useToggle(false);

  return (
    <div>
      <button onClick={toggle}>토글</button>
      {isOpen && <p>내용 표시</p>}
    </div>
  );
}

export default App;

Overwriting /content/bootcamp/src/App.jsx


In [57]:
%%writefile /content/bootcamp/src/App.jsx

import { useState } from 'react';

function useInput(initialValue = '') {
  const [value, setValue] = useState(initialValue);

  function onChange(e) {
    setValue(e.target.value);
  }

  return [value, onChange, setValue];
}

function App() {
  const [name, onChangeName] = useInput('');

  return (
    <div>
      <input value={name} onChange={onChangeName} />
      <p>{name}</p>
    </div>
  );
}

export default App;

Overwriting /content/bootcamp/src/App.jsx


In [58]:
%%bash
cd /content/bootcamp
npm install react-router-dom


up to date, audited 158 packages in 1s

38 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities


In [61]:
%%writefile /content/bootcamp/src/main.jsx

import { StrictMode } from 'react';
import { createRoot } from 'react-dom/client';
import App from './App.jsx';

createRoot(document.getElementById('root')).render(
  <StrictMode>
    <App />
  </StrictMode>
);

Overwriting /content/bootcamp/src/main.jsx


In [62]:
%%writefile /content/bootcamp/src/App.jsx

import { BrowserRouter, Routes, Route, Link } from 'react-router-dom';

function Home() {
  return <h1>홈 페이지입니다.</h1>;
}

function About() {
  return <h1>소개 페이지입니다.</h1>;
}

function App() {
  return (
    <BrowserRouter>
      <nav>
        <Link to="/">홈</Link> |
        <Link to="/about">소개</Link>
      </nav>

      <hr />

      <Routes>
        <Route path="/" element={<Home />} />
        <Route path="/about" element={<About />} />
      </Routes>
    </BrowserRouter>
  );
}

export default App;

Overwriting /content/bootcamp/src/App.jsx


In [63]:
%%writefile /content/bootcamp/src/App.jsx

import { useState, useEffect } from 'react';

function App() {
  const [data, setData] = useState([]);

  useEffect(() => {
    const loadData = async () => {
      try {
        const res = await fetch('https://jsonplaceholder.typicode.com/posts');
        const result = await res.json();
        setData(result);
      } catch (err) {
        console.error(err);
      }
    };

    loadData();
  }, []);

  return (
    <div>
      {data.map(item => (
        <p key={item.id}>{item.title}</p>
      ))}
    </div>
  );
}

export default App;

Overwriting /content/bootcamp/src/App.jsx


In [66]:
%%writefile /content/bootcamp/src/App.jsx

import { useState } from 'react';

function App() {
  const [title, setTitle] = useState('');

  const saveData = async () => {
    try {
      const res = await fetch('https://jsonplaceholder.typicode.com/posts', {
        method: 'POST',
        headers: {
          'Content-Type': 'application/json'
        },
        body: JSON.stringify({ title })
      });

      const result = await res.json();
      console.log(result);
    } catch (err) {
      console.error(err);
    }
  };

  return (
    <div>
      <input onChange={e => setTitle(e.target.value)} />
      <button onClick={saveData}>저장</button>
    </div>
  );
}

export default App;

Overwriting /content/bootcamp/src/App.jsx


In [67]:
%%writefile /content/bootcamp/src/App.jsx

import { useState, useEffect } from 'react';

function App() {
  const [data, setData] = useState([]);
  const [loading, setLoading] = useState(false);

  useEffect(() => {
    const loadData = async () => {
      try {
        setLoading(true);
        const res = await fetch('https://jsonplaceholder.typicode.com/posts');
        const result = await res.json();
        setData(result);
      } catch (err) {
        console.error(err);
      } finally {
        setLoading(false);
      }
    };

    loadData();
  }, []);

  return (
    <div>
      {loading ? <p>로딩 중...</p> : data.map(item => <p key={item.id}>{item.title}</p>)}
    </div>
  );
}

export default App;

Overwriting /content/bootcamp/src/App.jsx


In [68]:
%%writefile /content/bootcamp/src/App.jsx

import { BrowserRouter, Routes, Route, useParams } from 'react-router-dom';

function PostDetail() {
  const { id } = useParams();

  return <h1>{id}번 게시글</h1>;
}

function App() {
  return (
    <BrowserRouter>
      <Routes>
        <Route path="/posts/:id" element={<PostDetail />} />
      </Routes>
    </BrowserRouter>
  );
}

export default App;

Overwriting /content/bootcamp/src/App.jsx
